# Evaluación Sumativa 4 — Taller III
## Preprocesamiento, limpieza, transformación, validación y análisis de ventas con Apache Spark

**Nombre del notebook:** `mcdi502_s4_grupo3`  
**Grupo:** 3  
**Integrantes:** Eduardo Garrido · Luis Espinosa · Mauricio Ortega · Wilson Sebastián Arévalo Luna  
**Asignatura:** MCDI502  
**Entorno:** Databricks / PySpark / Spark SQL

### Distribución de funciones

- **Wilson Sebastián Arévalo Luna:** coordinación técnica, carga independiente de los archivos CSV, integración de los DataFrames y validación de los conteos por sucursal.
- **Luis Espinosa:** gestión de valores nulos, eliminación de duplicados y normalización de categorías, nombres de productos, fechas y precios unitarios.
- **Mauricio Ortega:** filtrado de registros inválidos, creación de columnas calculadas, validaciones de calidad y detección de outliers mediante IQR.
- **Eduardo Garrido:** creación de la vista temporal, consultas Spark SQL, interpretación de resultados, conclusiones y revisión de la documentación.
- **Responsabilidad conjunta:** revisión cruzada del código, ejecución integral mediante **Run All**, comprobación de resultados y preparación del archivo final de entrega.

### Objetivo
Consolidar las ventas del último trimestre de las sucursales de Viña del Mar, Santiago y Concepción; ejecutar un flujo distribuido de integración, limpieza, transformación y validación; y construir un dataset confiable para apoyar la toma de decisiones.

### Archivos utilizados
- `ventas_vina_del_mar.csv`
- `ventas_santiago.csv`
- `ventas_concepcion.csv`

### Preparación del entorno
Los tres archivos deben cargarse en una misma carpeta de Databricks. En Databricks Free Edition se recomienda un Volume, por ejemplo `/Volumes/workspace/default/ventas`. En un entorno que permita DBFS clásico se puede utilizar `dbfs:/FileStore/tables`. La celda siguiente permite editar la ruta sin cambiar el resto del Notebook.

# Parte 1 — Carga e integración de datos (ID3.1)

**Responsable principal:** Wilson Sebastián Arévalo Luna. **Revisión:** Eduardo Garrido, Luis Espinosa y Mauricio Ortega.

Se define un esquema explícito para conservar `fecha` y `precio_unitario` como texto durante la lectura. Esto evita interpretaciones incorrectas de fechas heterogéneas y precios que contienen símbolos monetarios. Cada CSV se carga por separado mediante `spark.read.csv`, tal como solicita la evaluación.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

# Ruta recomendada para Databricks Free Edition:
RUTA_BASE = "/Volumes/workspace/default/gestion-de-datos"

# Alternativa si el entorno permite DBFS clásico:
# RUTA_BASE = "dbfs:/FileStore/tables"

RUTA_BASE = RUTA_BASE.rstrip("/")

schema_ventas = StructType([
    StructField("id_transaccion", StringType(), True),
    StructField("fecha", StringType(), True),
    StructField("id_producto", StringType(), True),
    StructField("producto_nombre", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio_unitario", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("sucursal", StringType(), True)
])

ARCHIVOS = {
    "Viña del Mar": "ventas_vina_del_mar.csv",
    "Santiago": "ventas_santiago.csv",
    "Concepción": "ventas_concepcion.csv"
}


def cargar_csv(nombre_archivo):
    """Carga un CSV de ventas preservando los campos que requieren limpieza."""
    return (
        spark.read
        .option("header", True)
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")
        .schema(schema_ventas)
        .csv(f"{RUTA_BASE}/{nombre_archivo}")
    )


# Se cargan los tres archivos en DataFrames independientes.
df_vina = cargar_csv(ARCHIVOS["Viña del Mar"])
df_santiago = cargar_csv(ARCHIVOS["Santiago"])
df_concepcion = cargar_csv(ARCHIVOS["Concepción"])

print(f"Ruta utilizada: {RUTA_BASE}")


## Validación preventiva de las fuentes

Los archivos entregados contienen 105 registros de Viña del Mar, 125 de Santiago y 85 de Concepción. Los controles siguientes detienen la ejecución si un archivo está vacío, contaminado con otra sucursal o fue cargado desde una carpeta incorrecta. De esta manera se evita que la eliminación posterior de duplicados oculte una carga defectuosa.


In [ ]:
conteos_esperados = {
    "Viña del Mar": 105,
    "Santiago": 125,
    "Concepción": 85
}

dataframes_fuente = {
    "Viña del Mar": df_vina,
    "Santiago": df_santiago,
    "Concepción": df_concepcion
}

for sucursal_esperada, df_fuente in dataframes_fuente.items():
    total = df_fuente.count()
    sucursales_detectadas = [
        fila["sucursal"]
        for fila in df_fuente.select("sucursal").distinct().collect()
    ]

    print(
        f"{sucursal_esperada}: {total} registros; "
        f"sucursales detectadas: {sucursales_detectadas}"
    )

    assert total == conteos_esperados[sucursal_esperada], (
        f"Carga incorrecta para {sucursal_esperada}: se esperaban "
        f"{conteos_esperados[sucursal_esperada]} registros y se encontraron {total}. "
        f"Revise RUTA_BASE y el archivo CSV cargado."
    )

    assert sucursales_detectadas == [sucursal_esperada], (
        f"El archivo de {sucursal_esperada} contiene datos de otra sucursal: "
        f"{sucursales_detectadas}."
    )

print("Validación superada: las tres fuentes fueron cargadas correctamente.")


## Inspección inicial

Para cada DataFrame se muestra el esquema y las primeras cinco filas. Esta revisión permite observar formatos heterogéneos en `fecha`, símbolos en `precio_unitario`, categorías nulas y diferencias de escritura en los nombres de productos.


In [ ]:
print("=== VIÑA DEL MAR ===")
df_vina.printSchema()
df_vina.show(5, truncate=False)

print("=== SANTIAGO ===")
df_santiago.printSchema()
df_santiago.show(5, truncate=False)

print("=== CONCEPCIÓN ===")
df_concepcion.printSchema()
df_concepcion.show(5, truncate=False)


## Unión de DataFrames

Se utiliza `unionByName` para integrar las tres fuentes por nombre de columna. Después se comprueba que el total unificado sea igual a la suma de los registros originales.


In [ ]:
conteo_vina = df_vina.count()
conteo_santiago = df_santiago.count()
conteo_concepcion = df_concepcion.count()

# Se combinan los tres DataFrames conservando el nombre requerido por la rúbrica.
df_ventas_unificado = (
    df_vina
    .unionByName(df_santiago)
    .unionByName(df_concepcion)
)

conteo_esperado = conteo_vina + conteo_santiago + conteo_concepcion
conteo_unificado = df_ventas_unificado.count()

print(f"Registros Viña del Mar : {conteo_vina}")
print(f"Registros Santiago     : {conteo_santiago}")
print(f"Registros Concepción   : {conteo_concepcion}")
print(f"Total esperado         : {conteo_esperado}")
print(f"Total unificado        : {conteo_unificado}")

assert conteo_unificado == conteo_esperado == 315, (
    "La unión debe contener exactamente 315 registros: 105 + 125 + 85."
)

print("Validación correcta: el total unificado coincide con la suma de las sucursales.")


# Parte 2 — Limpieza y depuración de datos (ID4.2)

**Responsable principal:** Luis Espinosa. **Revisión:** Eduardo Garrido, Mauricio Ortega y Wilson Sebastián Arévalo Luna.

## Funciones para medir calidad

Se generan los indicadores mínimos solicitados: número total de filas, valores nulos o vacíos por columna y registros completamente duplicados. Se calcula el resumen antes de realizar cualquier limpieza.

In [ ]:
def expresiones_nulos(df):
    """Crea expresiones Spark para contar nulos y cadenas vacías por columna."""
    expresiones = []
    tipos = dict(df.dtypes)

    for columna in df.columns:
        if tipos[columna] == "string":
            condicion = F.col(columna).isNull() | (F.trim(F.col(columna)) == "")
        else:
            condicion = F.col(columna).isNull()

        expresiones.append(
            F.sum(F.when(condicion, 1).otherwise(0)).alias(columna)
        )

    return expresiones


def nulos_por_columna(df):
    return df.select(*expresiones_nulos(df))


def resumen_calidad(df, etapa):
    total_filas = df.count()
    duplicados = total_filas - df.dropDuplicates().count()
    detalle_nulos = nulos_por_columna(df).first().asDict()
    total_nulos = int(sum(valor or 0 for valor in detalle_nulos.values()))

    return spark.createDataFrame(
        [(etapa, total_filas, total_nulos, duplicados)],
        ["etapa", "total_filas", "total_nulos", "registros_duplicados"]
    )


calidad_antes = resumen_calidad(df_ventas_unificado, "Antes de la limpieza")

print("Resumen de calidad antes de la limpieza:")
display(calidad_antes)

print("Nulos por columna antes de la limpieza:")
display(nulos_por_columna(df_ventas_unificado))


## Gestión de nulos en `categoria`

Se opta por **imputar** mediante `.fillna({"categoria": "sin categoria"})` en vez de eliminar las ventas con `.dropna()`. En el dataset original existen 46 categorías nulas; eliminar esas filas provocaría pérdida de información comercial. Antes de aplicar `fillna`, las cadenas vacías se convierten a nulo para que reciban el mismo tratamiento.

## Eliminación de duplicados

Se utiliza `.dropDuplicates()` sobre todas las columnas, como solicita la actividad. El número de duplicados se calcula antes de eliminarlos.


In [ ]:
filas_antes_deduplicar = df_ventas_unificado.count()
df_sin_duplicados = df_ventas_unificado.dropDuplicates()
filas_despues_deduplicar = df_sin_duplicados.count()
duplicados_eliminados = filas_antes_deduplicar - filas_despues_deduplicar

print(f"Filas antes de eliminar duplicados : {filas_antes_deduplicar}")
print(f"Filas después de eliminar duplicados: {filas_despues_deduplicar}")
print(f"Duplicados eliminados               : {duplicados_eliminados}")

assert duplicados_eliminados == 15, (
    f"Se esperaban 15 duplicados completos y se detectaron {duplicados_eliminados}."
)


## Corrección de inconsistencias

Las transformaciones se implementan con funciones nativas de PySpark:

- `lower`, `trim` y `regexp_replace` para normalizar textos.
- `to_date` y expresiones regulares para convertir los cuatro formatos observados en `fecha`.
- `regexp_replace` para retirar `$`, espacios y otros símbolos de `precio_unitario`.
- Conversión final de `precio_unitario` a `double`.
- `.fillna()` para imputar la categoría faltante.


In [ ]:
# Las cadenas vacías de categoria se convierten primero a nulo.
df_categoria_preparada = df_sin_duplicados.withColumn(
    "categoria",
    F.when(
        F.col("categoria").isNull() | (F.trim(F.col("categoria")) == ""),
        F.lit(None).cast("string")
    ).otherwise(F.col("categoria"))
)

# Imputación explícita exigida por la rúbrica.
df_categoria_imputada = df_categoria_preparada.fillna(
    {"categoria": "sin categoria"}
)

fecha_texto = F.trim(F.col("fecha"))
fecha_estandarizada = (
    F.when(fecha_texto.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(fecha_texto, "yyyy-MM-dd"))
     .when(fecha_texto.rlike(r"^\d{4}/\d{2}/\d{2}$"), F.to_date(fecha_texto, "yyyy/MM/dd"))
     .when(fecha_texto.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(fecha_texto, "dd/MM/yyyy"))
     .when(fecha_texto.rlike(r"^\d{2}-\d{2}-\d{4}$"), F.to_date(fecha_texto, "dd-MM-yyyy"))
     .otherwise(F.lit(None).cast("date"))
)

# Los precios corresponden a pesos chilenos enteros; se conservan dígitos y signo negativo.
precio_limpio_texto = F.regexp_replace(
    F.trim(F.col("precio_unitario")),
    r"[^0-9-]",
    ""
)

df_normalizado = (
    df_categoria_imputada
    .withColumn(
        "producto_nombre",
        F.lower(F.regexp_replace(F.trim(F.col("producto_nombre")), r"\s+", " "))
    )
    .withColumn(
        "categoria",
        F.lower(F.regexp_replace(F.trim(F.col("categoria")), r"\s+", " "))
    )
    .withColumn("sucursal", F.trim(F.col("sucursal")))
    .withColumn("id_transaccion", F.trim(F.col("id_transaccion")))
    .withColumn("id_producto", F.trim(F.col("id_producto")))
    .withColumn("id_cliente", F.trim(F.col("id_cliente")))
    .withColumn("fecha", fecha_estandarizada)
    .withColumn(
        "precio_unitario",
        F.when(
            F.length(precio_limpio_texto) > 0,
            precio_limpio_texto.cast("double")
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn("cantidad", F.col("cantidad").cast("integer"))
)

df_normalizado.printSchema()
df_normalizado.show(10, truncate=False)


## Validación de las conversiones

Se comprueba si alguna fecha, precio o cantidad no pudo convertirse. Una conversión fallida puede generar un nulo nuevo, por lo que debe medirse antes del filtrado.


In [ ]:
conversiones_fallidas = df_normalizado.select(
    F.sum(F.when(F.col("fecha").isNull(), 1).otherwise(0)).alias("fechas_no_convertidas"),
    F.sum(F.when(F.col("precio_unitario").isNull(), 1).otherwise(0)).alias("precios_no_convertidos"),
    F.sum(F.when(F.col("cantidad").isNull(), 1).otherwise(0)).alias("cantidades_no_convertidas")
)

display(conversiones_fallidas)

fila_conversiones = conversiones_fallidas.first()
assert fila_conversiones["fechas_no_convertidas"] == 0
assert fila_conversiones["precios_no_convertidos"] == 0
assert fila_conversiones["cantidades_no_convertidas"] == 0


# Parte 3 — Transformación y análisis con DataFrames (ID4.1)

**Responsable principal:** Mauricio Ortega. **Revisión:** Eduardo Garrido, Luis Espinosa y Wilson Sebastián Arévalo Luna.

## Filtrado de datos inválidos

Una venta válida requiere una cantidad estrictamente mayor que cero. Los registros inválidos se mantienen en un DataFrame de auditoría y se excluyen del conjunto analítico mediante `.filter()`.

In [ ]:
df_cantidades_invalidas = df_normalizado.filter(
    F.col("cantidad").isNull() | (F.col("cantidad") <= 0)
)

cantidad_registros_invalidos = df_cantidades_invalidas.count()
print(f"Registros con cantidad nula, cero o negativa: {cantidad_registros_invalidos}")
display(df_cantidades_invalidas.orderBy("sucursal", "id_transaccion"))

assert cantidad_registros_invalidos == 26, (
    f"Se esperaban 26 cantidades inválidas y se detectaron {cantidad_registros_invalidos}."
)

df_filtrado = df_normalizado.filter(
    (F.col("cantidad") > 0) &
    F.col("precio_unitario").isNotNull() &
    F.col("fecha").isNotNull()
)

print(f"Registros después del filtrado básico: {df_filtrado.count()}")
assert df_filtrado.count() == 274


## Creación de columnas calculadas

- `precio_total`: resultado de `cantidad * precio_unitario`.
- `dia_semana`: día correspondiente a la fecha estandarizada.


In [ ]:
# dayofweek: 1=domingo, 2=lunes, ..., 7=sábado.
mapa_dias = F.create_map(
    F.lit(1), F.lit("domingo"),
    F.lit(2), F.lit("lunes"),
    F.lit(3), F.lit("martes"),
    F.lit(4), F.lit("miércoles"),
    F.lit(5), F.lit("jueves"),
    F.lit(6), F.lit("viernes"),
    F.lit(7), F.lit("sábado")
)

df_transformado = (
    df_filtrado
    .withColumn(
        "precio_total",
        F.round(F.col("cantidad") * F.col("precio_unitario"), 2)
    )
    .withColumn(
        "dia_semana",
        mapa_dias[F.dayofweek(F.col("fecha"))]
    )
)

df_transformado.select(
    "id_transaccion", "fecha", "dia_semana", "cantidad",
    "precio_unitario", "precio_total"
).show(10, truncate=False)


## Análisis con `distinct()`

Se cuentan y muestran las sucursales y categorías únicas después de la limpieza y el filtrado básico.


In [ ]:
sucursales_unicas = df_transformado.select("sucursal").distinct()
categorias_unicas = df_transformado.select("categoria").distinct()

print(f"Cantidad de sucursales únicas: {sucursales_unicas.count()}")
display(sucursales_unicas.orderBy("sucursal"))

print(f"Cantidad de categorías únicas: {categorias_unicas.count()}")
display(categorias_unicas.orderBy("categoria"))

assert sucursales_unicas.count() == 3


# Parte 4 — Validación y control de calidad (ID4.3)

**Responsable principal:** Mauricio Ortega. **Revisión:** Eduardo Garrido, Luis Espinosa y Wilson Sebastián Arévalo Luna.

## Regla 1: `precio_total` no puede ser negativo

Se identifican los incumplimientos y se conserva solamente el conjunto que cumple la regla.

In [ ]:
df_totales_negativos = df_transformado.filter(F.col("precio_total") < 0)
numero_totales_negativos = df_totales_negativos.count()

print(f"Registros con precio_total negativo: {numero_totales_negativos}")
display(df_totales_negativos)

df_validado_base = df_transformado.filter(F.col("precio_total") >= 0)

assert numero_totales_negativos == 0


## Regla 2: detección de outliers mediante IQR

Se calcula el rango intercuartílico de `precio_unitario`:

- `IQR = Q3 - Q1`
- Límite inferior: `Q1 - 1,5 × IQR`
- Límite superior: `Q3 + 1,5 × IQR`

Los registros fuera de los límites se marcan con `es_outlier = true` y se separan en `df_outliers`. El conjunto `df_ventas_final` excluye los outliers para evitar que distorsionen las consultas analíticas, pero estos permanecen disponibles para auditoría.


In [ ]:
q1, q3 = df_validado_base.approxQuantile(
    "precio_unitario",
    [0.25, 0.75],
    0.0
)

iqr = q3 - q1
limite_inferior = q1 - (1.5 * iqr)
limite_superior = q3 + (1.5 * iqr)

print(f"Q1              : {q1:,.2f}")
print(f"Q3              : {q3:,.2f}")
print(f"IQR             : {iqr:,.2f}")
print(f"Límite inferior : {limite_inferior:,.2f}")
print(f"Límite superior : {limite_superior:,.2f}")

condicion_outlier = (
    (F.col("precio_unitario") < F.lit(limite_inferior)) |
    (F.col("precio_unitario") > F.lit(limite_superior))
)

df_con_marca_outlier = df_validado_base.withColumn(
    "es_outlier",
    condicion_outlier
)

df_outliers = df_con_marca_outlier.filter(F.col("es_outlier"))
df_ventas_final = df_con_marca_outlier.filter(~F.col("es_outlier"))

cantidad_outliers = df_outliers.count()
cantidad_final = df_ventas_final.count()

print(f"Registros marcados como outlier : {cantidad_outliers}")
print(f"Registros del dataset final     : {cantidad_final}")

assert q1 == 79990.0
assert q3 == 299990.0
assert limite_superior == 629990.0
assert cantidad_outliers == 7
assert cantidad_final == 267

display(
    df_outliers.select(
        "id_transaccion", "sucursal", "producto_nombre",
        "cantidad", "precio_unitario", "precio_total", "es_outlier"
    ).orderBy(F.desc("precio_unitario"))
)


### Revisión complementaria de precios muy bajos

El límite inferior del IQR es negativo, por lo que precios positivos como 1 o 50 no se clasifican estadísticamente como outliers. Sin una regla comercial que defina un precio mínimo, estos registros no se eliminan automáticamente y se muestran para revisión.


In [ ]:
df_precios_muy_bajos = df_ventas_final.filter(F.col("precio_unitario") <= 50)

print(
    "Registros con precio_unitario menor o igual a 50: "
    f"{df_precios_muy_bajos.count()}"
)

display(
    df_precios_muy_bajos.select(
        "id_transaccion", "sucursal", "producto_nombre", "precio_unitario"
    ).orderBy("precio_unitario")
)


## Indicadores de calidad antes y después

Se vuelve a calcular el número de filas, los nulos por columna y los duplicados sobre el dataset final. La comparación permite demostrar objetivamente el efecto del proceso de limpieza.


In [ ]:
calidad_despues = resumen_calidad(df_ventas_final, "Después de la limpieza")
resumen_comparativo = calidad_antes.unionByName(calidad_despues)

display(resumen_comparativo)

print("Nulos por columna después de la limpieza:")
display(nulos_por_columna(df_ventas_final))

# Controles finales automatizados.
assert df_ventas_final.filter(F.col("precio_total") < 0).count() == 0
assert df_ventas_final.count() == df_ventas_final.dropDuplicates().count()
assert sum(nulos_por_columna(df_ventas_final).first()) == 0
assert df_ventas_final.count() == 267

print("Controles finales superados: sin totales negativos, sin duplicados y sin nulos.")


# Parte 5 — Análisis con Spark SQL (ID3.3)

**Responsable principal:** Eduardo Garrido. **Revisión:** Luis Espinosa, Mauricio Ortega y Wilson Sebastián Arévalo Luna.

El DataFrame limpio y validado se registra como vista temporal con el nombre exigido `ventas_limpias`. Los outliers permanecen en `df_outliers` para auditoría, pero no se incorporan a la vista analítica.

In [ ]:
df_ventas_final.createOrReplaceTempView("ventas_limpias")

print("Vista temporal creada correctamente: ventas_limpias")
spark.sql("DESCRIBE ventas_limpias").show(truncate=False)


## Consulta 1 — Diez ventas con mayor `precio_total`

La consulta identifica las transacciones individuales de mayor valor, mostrando exactamente las columnas solicitadas.


In [ ]:
%sql
SELECT
    id_transaccion,
    fecha,
    sucursal,
    producto_nombre,
    precio_total
FROM ventas_limpias
ORDER BY precio_total DESC
LIMIT 10


**Hallazgo esperado:** el mayor `precio_total` limpio es 4.499.900 y aparece en dos transacciones. Las ventas superiores corresponden principalmente a productos tecnológicos vendidos en cantidades altas.


## Consulta 2 — Total de ventas y cantidad de transacciones por sucursal

La suma de `precio_total` permite comparar el desempeño monetario, mientras que el conteo representa la cantidad de transacciones válidas después del proceso de calidad.


In [ ]:
%sql
SELECT
    sucursal,
    ROUND(SUM(precio_total), 2) AS total_ventas,
    COUNT(*) AS cantidad_transacciones
FROM ventas_limpias
GROUP BY sucursal
ORDER BY total_ventas DESC


**Hallazgo esperado:** Santiago lidera con 108.320.664 en ventas limpias y 108 transacciones; Viña del Mar registra 89.038.895 y 91 transacciones; y Concepción alcanza 56.110.510 y 68 transacciones.


## Consulta 3 — Precio promedio, mínimo y máximo por categoría

La consulta resume el comportamiento del precio unitario y permite comparar el nivel y la dispersión de precios de cada categoría.


In [ ]:
%sql
SELECT
    categoria,
    ROUND(AVG(precio_unitario), 2) AS precio_promedio,
    ROUND(MIN(precio_unitario), 2) AS precio_minimo,
    ROUND(MAX(precio_unitario), 2) AS precio_maximo
FROM ventas_limpias
GROUP BY categoria
ORDER BY precio_promedio DESC


**Hallazgo esperado:** `almacenamiento` presenta el mayor precio promedio entre las categorías identificadas. Los valores positivos muy bajos deben revisarse con una regla comercial adicional, porque no son detectados por el límite inferior del IQR.


# Parte 6 — Conclusiones y trabajo grupal

**Responsable principal de consolidación:** Eduardo Garrido.  
**Revisión técnica conjunta:** Luis Espinosa, Mauricio Ortega y Wilson Sebastián Arévalo Luna.

## Resultados esperados con los archivos entregados

- Registros originales: **315**.
- Registros de Viña del Mar: **105**.
- Registros de Santiago: **125**.
- Registros de Concepción: **85**.
- Categorías nulas antes de limpiar: **46**.
- Registros completamente duplicados: **15**.
- Registros únicos después de deduplicar: **300**.
- Cantidades nulas, cero o negativas: **26**.
- Fechas que no pudieron convertirse: **0**.
- Precios que no pudieron convertirse: **0**.
- Registros válidos antes del control IQR: **274**.
- Q1 de `precio_unitario`: **79.990**.
- Q3 de `precio_unitario`: **299.990**.
- Límite superior del IQR: **629.990**.
- Outliers detectados: **7**.
- Registros del dataset analítico final: **267**.
- Nulos finales: **0**.
- Duplicados finales: **0**.
- Ventas limpias totales: **253.470.069**.

## Interpretación

El proceso permitió integrar las tres fuentes sin mezclar sucursales, corregir formatos heterogéneos, conservar ventas con categoría desconocida mediante imputación, eliminar duplicados, excluir cantidades inválidas y separar precios extremos. Santiago presenta el mayor total de ventas válidas. El análisis también demuestra que una regla estadística debe complementarse con conocimiento del negocio: el IQR detecta los precios extremadamente altos, pero no necesariamente los valores positivos anormalmente bajos.

## Evidencia de trabajo grupal

- **Carga e integración de fuentes:** Wilson Sebastián Arévalo Luna.
- **Limpieza, depuración y estandarización:** Luis Espinosa.
- **Transformación, validación, indicadores de calidad e IQR:** Mauricio Ortega.
- **Spark SQL, análisis de resultados, conclusiones y documentación:** Eduardo Garrido.
- **Revisión cruzada del Notebook:** Eduardo Garrido, Luis Espinosa, Mauricio Ortega y Wilson Sebastián Arévalo Luna.
- **Ejecución integral, validación de cifras y preparación de la entrega:** responsabilidad conjunta de los cuatro integrantes.
- **Fecha de consolidación del trabajo:** 2 de agosto de 2026.

# Referencias

- Material de la Unidad 2 / Semana 3 del curso MCDI502.
- Apache Spark. *PySpark DataFrame API Reference*: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html
- Apache Spark. *Built-in SQL Functions*: https://spark.apache.org/docs/latest/sql-ref-functions-builtin.html

> La revisión del código, la comprensión de las decisiones de limpieza, la ejecución mediante **Run All**, la validación de resultados y la exportación del Notebook con sus salidas corresponden conjuntamente a Eduardo Garrido, Luis Espinosa, Mauricio Ortega y Wilson Sebastián Arévalo Luna.